# 比较 CACE 当前 Qeq 求解 与 DP-QEq 式 LBFGS 加速

本 notebook 在相同构型与参数下对比两种方式：
1. **当前 CACE**：构造 $A$ 矩阵后对增广线性系统 `torch.linalg.solve` 直接求 $q$；
2. **DP-QEq 式**：不构造 $A$，用“能量 + 投影梯度 + LBFGS”迭代求 $q$。

比较内容：电荷 $q$ 的差异、Qeq 能量、以及耗时。

In [ ]:
# 路径与导入：使 notebook 能导入 cace 包（请根据你本机路径调整）
import sys
from pathlib import Path

# 假设 notebook 在 SOG-Qeq/SOG-Net/CACE-SOG-Qeq，cace 包在 SOG-Qeq/cace
# 从当前目录往上是 SOG-Net，再往上是 SOG-Qeq，cace 在 SOG-Qeq/cace
for _cand in [Path("../../cace").resolve(), Path("..").resolve() / "cace", Path(".").resolve().parent.parent / "cace"]:
    if _cand.exists() and (_cand / "cace").is_dir():
        sys.path.insert(0, str(_cand))
        break
else:
    sys.path.insert(0, str(Path("../..").resolve() / "cace"))

import torch
from cace.modules.charge_eq import ChargeEq
from cace.modules.ewald import EwaldPotential

print("PyTorch:", torch.__version__)
print("CACE ChargeEq / Ewald 导入成功")

: 

In [ ]:
# 构造单构型测试数据：坐标 r、晶胞 cell、电负性 chi、硬度 J、总电荷 Q_tot
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
torch.manual_seed(42)

# 原子数（可改大以观察加速效果）
N = 32
# 正交晶胞 (Å)
L = 12.0
cell = torch.eye(3, device=device, dtype=dtype) * L
# 随机坐标（在盒内）
r = torch.rand(N, 3, device=device, dtype=dtype) * L * 0.8
# 电负性、硬度（示例：两种元素交替）
elements = [1, 8]  # H, O
Z = torch.tensor([elements[i % 2] for i in range(N)], device=device, dtype=torch.long)
chi = torch.randn(N, device=device, dtype=dtype) * 0.5 + 5.0  # 大致 4.5--5.5
# ChargeEq 用 J_raw^2 得到 J，这里直接给每原子 J
J = torch.ones(N, device=device, dtype=dtype) * 1.0
Q_tot = 0.0  # 中性体系

# 构造 ChargeEq 模块以复用其 Ewald 与归一化
charge_eq = ChargeEq(
    dl=1.5, sigma=1.0, elements=elements,
    system_charge=Q_tot, remove_self_interaction=True, compute_field=True
).to(device=device, dtype=dtype)
charge_eq.eval()

# 归一化后的总电荷（与 _compute_q_eq 一致）
Q_tot_norm = Q_tot / charge_eq.normalization_factor
print("N =", N, "  Q_tot_norm =", Q_tot_norm.item())

In [ ]:
# 从 ChargeEq 取出与 Z 对应的硬度 J（与 forward 中一致）
with torch.no_grad():
    J_elem = torch.square(charge_eq.J_raw)
    J = J_elem[charge_eq.Z_index_map[Z]].to(dtype).to(device)
print("J 范围:", J.min().item(), "~", J.max().item())

## 方法 1：当前 CACE — 构造 $A$ 后直接求逆

In [ ]:
# 方法 1：构造 A，再 torch.linalg.solve 求 q
import time
t0 = time.perf_counter()
with torch.no_grad():
    A_mat = charge_eq._compute_A_matrix(r, cell)
    # chi 传 2D 避免 _compute_q_eq 内 RHS 被覆盖
    q_direct, lambda_direct = charge_eq._compute_q_eq(A_mat, chi.unsqueeze(1), J, Q_tot)
    q_direct = q_direct.view(-1)
t_direct = time.perf_counter() - t0
# Qeq 能量 0.5 q'Aq + 0.5 J_i q_i^2 + chi_i q_i
E_direct = (0.5 * torch.dot(q_direct, torch.mv(A_mat, q_direct)) + 0.5 * (J * q_direct * q_direct).sum() + (chi * q_direct).sum()).item()
print("直接求逆: 耗时 = {:.4f} s".format(t_direct))
print("q 范围: {:.4f} ~ {:.4f}, sum(q) = {:.6f}".format(q_direct.min().item(), q_direct.max().item(), q_direct.sum().item()))
print("E_Qeq (直接) = {:.6f}".format(E_direct))

## 方法 2：DP-QEq 式 — 不构造 $A$，投影梯度 + LBFGS

In [ ]:
def energy_and_grad_q(q, r, cell, chi, J, ep, norm_factor):
    """E_Qeq(q) 与 grad E_Qeq(q)，不构造 A；库仑部分用 Ewald 一次调用得到 E 与 Aq。"""
    q = q.reshape(-1, 1)
    pot, q_field = ep.compute_potential_triclinic(r, q, cell, compute_field=True)
    E_coul = pot.squeeze()
    Aq = q_field.squeeze(1)
    E_on = (chi * q.squeeze() + 0.5 * J * q.squeeze() ** 2).sum()
    E = E_coul + E_on
    grad = Aq + J * q.squeeze() + chi
    return E, grad

def project_grad(grad, one_vec):
    """投影梯度到 1'q=const 的切空间：g <- g - (1'g/n) 1"""
    n = one_vec.numel()
    a = one_vec.dot(grad)
    return grad - (a / n) * one_vec

def solve_q_lbfgs(r, cell, chi, J, Q_tot_norm, ep, max_iter=200, tol=1e-6, init_q=None):
    N = r.shape[0]
    one = torch.ones(N, device=r.device, dtype=r.dtype)
    if init_q is None:
        q = torch.full((N,), Q_tot_norm / N, device=r.device, dtype=r.dtype)
    else:
        q = init_q.clone().detach()
        q = q - (q.sum() - Q_tot_norm) / N
    q = q.requires_grad_(True)
    optimizer = torch.optim.LBFGS([q], lr=1.0, max_iter=20, line_search_fn="strong_wolfe")
    def closure():
        optimizer.zero_grad()
        with torch.no_grad():
            E, grad = energy_and_grad_q(q, r, cell, chi, J, ep, charge_eq.normalization_factor)
        grad_proj = project_grad(grad, one)
        q.grad = grad_proj.clone()
        return E
    for _ in range(max_iter // 20):
        optimizer.step(closure)
        with torch.no_grad():
            q.data.sub_((q.sum() - Q_tot_norm) / N)
    with torch.no_grad():
        E_final, _ = energy_and_grad_q(q, r, cell, chi, J, ep, charge_eq.normalization_factor)
    return q.detach(), E_final.item()

In [ ]:
# 方法 2：LBFGS 求解（以直接解作为初猜可加快收敛）
t0 = time.perf_counter()
q_lbfgs, E_lbfgs = solve_q_lbfgs(
    r, cell, chi, J, Q_tot_norm, charge_eq.ep,
    max_iter=200, tol=1e-6, init_q=q_direct.detach().clone()
)
t_lbfgs = time.perf_counter() - t0
print("LBFGS: 耗时 = {:.4f} s".format(t_lbfgs))
print("q 范围: {:.4f} ~ {:.4f}, sum(q) = {:.6f}".format(q_lbfgs.min().item(), q_lbfgs.max().item(), q_lbfgs.sum().item()))
print("E_Qeq (LBFGS) = {:.6f}".format(E_lbfgs))

## 比较结果

In [ ]:
# 电荷差异与能量差异
diff = (q_direct - q_lbfgs).detach()
print("--- 电荷 q 比较 ---")
print("  max |q_direct - q_lbfgs| = {:.2e}".format(diff.abs().max().item()))
print("  mean |q_direct - q_lbfgs| = {:.2e}".format(diff.abs().mean().item()))
print("--- 能量比较 ---")
print("  E_direct = {:.6f},  E_lbfgs = {:.6f}".format(E_direct, E_lbfgs))
print("  |E_direct - E_lbfgs| = {:.2e}".format(abs(E_direct - E_lbfgs)))
print("--- 耗时 ---")
print("  直接求逆: {:.4f} s,  LBFGS: {:.4f} s".format(t_direct, t_lbfgs))

In [ ]:
# 简单图示：直接解 vs LBFGS 的电荷
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(q_direct.cpu().numpy(), label="直接求逆 $q$", alpha=0.8)
ax.plot(q_lbfgs.cpu().numpy(), label="LBFGS $q$", alpha=0.8, linestyle="--")
ax.set_xlabel("原子索引")
ax.set_ylabel("电荷 $q$")
ax.legend()
ax.set_title("两种方法得到的 Qeq 电荷对比")
plt.tight_layout()
plt.show()

In [ ]:
# 可选：无初猜时 LBFGS 收敛（init_q=None，从均匀 q=Q_tot/N 开始）
q_lbfgs_cold, E_lbfgs_cold = solve_q_lbfgs(
    r, cell, chi, J, Q_tot_norm, charge_eq.ep,
    max_iter=400, tol=1e-6, init_q=None
)
print("无初猜 LBFGS: E = {:.6f}, max|q - q_direct| = {:.2e}".format(
    E_lbfgs_cold, (q_lbfgs_cold - q_direct).abs().max().item()))